# Labeled data full set

In [ ]:
import pandas as pd

In [ ]:
# import labeled data ( random 1000 posts lebeled by Elvie and  - 100 double checked by me )
aba_posts_labeled1 = pd.read_csv('data/labeled_data/for_elvie_labeled.csv')
aba_posts_labeled1.info()
print()

aba_posts_labeled2 = pd.read_csv('data/labeled_data/for_elvie2_labeled.csv')
aba_posts_labeled2.info()
print()

aba_posts_labeled3 = pd.read_csv('data/labeled_data/for_elvie3_labeled.csv')
aba_posts_labeled3.info()
print()

aba_posts_labeled4 = pd.read_csv('data/labeled_data/for_elvie4_labeled.csv')
aba_posts_labeled4.info()
print()

aba_posts_labeled5 = pd.read_csv('data/labeled_data/for_elvie5_labeled.csv')
aba_posts_labeled5.info()
print()

aba_posts_labeled6 = pd.read_csv('data/labeled_data/for_maria_labeled.csv')
aba_posts_labeled6.info()
print()

aba_posts_labeled7 = pd.read_csv('data/labeled_data/for_andrea_labeled.csv')
aba_posts_labeled7.info()

In [ ]:
# join the labeled data ( 1000 random posts )
aba_posts_labeled = pd.concat([aba_posts_labeled1, aba_posts_labeled2, aba_posts_labeled3, aba_posts_labeled4, aba_posts_labeled5, aba_posts_labeled6, aba_posts_labeled7], ignore_index=True)
aba_posts_labeled.info()

In [ ]:
# Rename columns as needed
column_mapping = {
    'Sentiment (poz / neg/ neutral)': 'sentiment',
    'Author (can be parent/carer/professions/person with ASD) leave blank if it is not clearly stated)': 'author',
    'Comments - anything you deem important info': 'commnents'
}

aba_posts_labeled.rename(columns=column_mapping, inplace=True)
aba_posts_labeled.info()

In [ ]:
aba_posts_labeled.head()

In [ ]:
# 104 entries sentiment col missing
#?sentiment was not recorded as the posts were not relevant to aba - see comments
# remove these rows
# the bertopic model resulted in almost all posts being relevant to aba - only about 100 out of 3500 posts are not aba related (less than 1%)
aba_posts_labeled[aba_posts_labeled['sentiment'].isnull()]

In [ ]:
aba_posts_labeled = aba_posts_labeled.dropna(subset=['sentiment'])
aba_posts_labeled.info()

In [ ]:
# as per instruction given for labling atuhor entry was left blank if the author was not clear in the post
# replace the NaN values in authot with 'layperson' 
aba_posts_labeled['author'].fillna('layperson', inplace=True)
aba_posts_labeled.info()

In [ ]:
# look at sentiment classes 
# there are some misspelling -> can be corrected
# sentiment is blank space in 4 rows
aba_posts_labeled['sentiment'].value_counts()

In [ ]:
empty_sentiment_rows = aba_posts_labeled[aba_posts_labeled['sentiment'].str.isspace()] 
empty_sentiment_rows  # posts not about aba -> drop them

In [ ]:
aba_posts_labeled.drop(empty_sentiment_rows.index, inplace=True)
aba_posts_labeled.info()
# 3452 posts to be inlcuded in the analysis 
# comments columns has missing data - however this col. is only for reference and will be dropped for final analysis

In [ ]:
# check rows where sentiment contains a ? ( grader was unsure )
question_mark_rows = aba_posts_labeled[aba_posts_labeled['sentiment'].str.contains('\?', na=False, regex=True)]
question_mark_rows

In [ ]:
# if the coders were confused the ML algo. might be as well 
# drop rows where sentiment contains '?'
aba_posts_labeled = aba_posts_labeled[~aba_posts_labeled['sentiment'].str.contains('\?')]
aba_posts_labeled.info()

# 3449 rows / posts remain

## Sentiment 
descriptive statistics

In [ ]:
aba_posts_labeled['sentiment'].value_counts()  # category labeles need cleaning ( spelling checks and consistency )

In [ ]:
# drop Ne row - see reasoning in file: sentiment_analysis_aba_posts.ipynb
aba_posts_labeled = aba_posts_labeled[aba_posts_labeled['sentiment'] != 'Ne']
# Replace 'Pozitive Negative' with 'Neutral' in the 'sentiment' column
aba_posts_labeled['sentiment'] = aba_posts_labeled['sentiment'].replace('Pozitive Negative', 'Neutral')
# remove whitespace 
aba_posts_labeled['sentiment'] = aba_posts_labeled['sentiment'].str.strip()

aba_posts_labeled.info()

In [ ]:
aba_posts_labeled['sentiment'].value_counts() 

In [ ]:
# make spelling uniform for pozitive sentiment label 
mask = aba_posts_labeled['sentiment'].str.startswith('Po')
aba_posts_labeled.loc[mask, 'sentiment'] = 'Positive'
aba_posts_labeled['sentiment'].value_counts() 

In [ ]:
import matplotlib.pyplot as plt

# Plot the 'sentiment' column
sentiment_counts = aba_posts_labeled['sentiment'].value_counts()
sentiment_counts.plot(kind='bar', color='skyblue')
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of posts')
plt.show()

# more than half the posts present as neutral regarding aba therapy 
# data is not balanced
# the data distribution is similar to the smaller data set (used in file: sentiment_analysis_aba_posts.ipynb)

# as per instructions and also by inspecting the comments column :
# posts that are neutral are either:
#   - information/advice seeking about aba therapy related
#   - information/advice seeking about aba job related
#   - information/advice seeking about aba in general
#   - people expressing both pozitive and negative opinions about aba in the same post 

# negative sentiments were recorded for those posts that:
#   - expressed only opinions against aba therapy or aba in general
#   - expressed only opinions against aba as a career field/ job related 

# pozitive sentiments were recorded for those posts that:
#   - expressed only pozitive opinions about aba therapy or aba in general
#   - expressed only pozitive opinions about aba as a career field/ job related 

## Post's Author
descriptive statistics

In [ ]:
# look at categories of authors
# strip white spaces
aba_posts_labeled['author'] = aba_posts_labeled['author'].str.strip()
# lowewrcase all lables
aba_posts_labeled['author'] = aba_posts_labeled['author'].str.lower()
aba_posts_labeled['author'].value_counts() 

In [ ]:
# replace ? containing labels with layperson ( as the grader was not sure , it means it is not clear in the text)
question_mark_rows = aba_posts_labeled[aba_posts_labeled['author'].str.contains('\?', na=False, regex=True)]
aba_posts_labeled.loc[question_mark_rows.index, 'author'] = 'layperson'
aba_posts_labeled['author'].value_counts()

In [ ]:
# replace other people with layperson 
other_people = ['relativeo of aba professional', 'student', 'aba student', 'person with ?special needs? not specified']

other_people_mapping = {person: 'layperson' for person in other_people}
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace(other_people_mapping)

aba_posts_labeled.author.value_counts()

In [ ]:
# replace other therapy/education related professions with other_professions
other_professions = ['slp', 'ot', 'salt', 'psychologist', 'teacher', 'education professional', 'professors', 'occupational therapist', 'professional', 'ex aba professional / ot']

profession_mapping = {profession: 'other_professional' for profession in other_professions}

aba_posts_labeled['author'] = aba_posts_labeled['author'].replace(profession_mapping)

aba_posts_labeled.author.value_counts()

In [ ]:
# replace other therapy/education related professions with other_professions
family = ['sibling', 'friend of asd person', 'relative']

family_mapping = {member: 'family' for member in family}

aba_posts_labeled['author'] = aba_posts_labeled['author'].replace(family_mapping)

aba_posts_labeled.author.value_counts()

In [ ]:
# standardize labels for ex aba professional
ex_aba_professional = ['ex aba professional', 'ex - aba professional']

ex_aba_mapping = {member: 'ex aba profesisonal' for member in ex_aba_professional}

aba_posts_labeled['author'] = aba_posts_labeled['author'].replace(ex_aba_mapping)

aba_posts_labeled.author.value_counts()

In [ ]:
# standardize labels for person with ?special needs? not specified ( = layperson) and parent
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('asd student', 'asd person')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('parent with asd child', 'parent')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('other professional', 'other_professional')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('asd person', 'asd')


aba_posts_labeled.author.value_counts()

In [ ]:
# replace other therapy/education related professions with other_professions in strings
other_professions = [ 'ot', 'psychologist', 'teacher', 'social worker', 'clinical therapist']
profession_mapping = {profession: 'other_professional' for profession in other_professions}

aba_posts_labeled['author'] = aba_posts_labeled['author'].replace(profession_mapping, regex=True)

aba_posts_labeled.author.value_counts()

In [ ]:
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('other_professionalher_professional', 'other_professional')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('other_professionalher_professional', 'other_professional', regex=True)
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('other_professionalher professional', 'other_professional', regex=True)

aba_posts_labeled.author.value_counts()

In [ ]:
# repalce 'with' with '/'
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('with', '/', regex=True)
aba_posts_labeled.author.value_counts()

In [ ]:
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('asd person', 'asd')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('asd parent', 'parent / asd')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('asd other_professional', 'asd / other_professional')

aba_posts_labeled.author.value_counts()

In [ ]:
# family - > move to layperson as they are not directly involved in decission making regarding the therapy
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('family', 'layperson')
# ex aba proffessionals go into other_professionals as they are not currently practicing aba
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('ex aba profesisonal', 'other_professional')
aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('ex aba profesional', 'other_professional')

aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('aba student / asd', 'asd')

aba_posts_labeled['author'] = aba_posts_labeled['author'].replace('aba  professional/asd', 'aba professional/asd')

aba_posts_labeled.author.value_counts()

In [ ]:
# split the 'author' column and convert to lists
aba_posts_labeled['author'] = aba_posts_labeled['author'].str.split('/')
# strip white spaces from each element within the 'author' column
aba_posts_labeled['author'] = aba_posts_labeled['author'].apply(lambda x: [i.strip() for i in x])
aba_posts_labeled.author.value_counts()

In [ ]:
# some list have the same entries but reversed orther -> make it uniform
aba_posts_labeled['author'] = aba_posts_labeled['author'].apply(set)
aba_posts_labeled['author'] = aba_posts_labeled['author'].apply(list)
aba_posts_labeled.author.value_counts()

In [ ]:
aba_posts_labeled.info()  # 3448 rows/ post to include in sentiment analisys

In [ ]:
# Count all the labels per category
aba_posts_labeled["author"].explode().value_counts()  

In [ ]:
# Get unique items & counts
unique_values = aba_posts_labeled["author"].explode().value_counts().index.tolist()
unique_value_counts = aba_posts_labeled["author"].explode().value_counts().values

# Create plot
fig, ax = plt.subplots(figsize = (10,6))
ax.bar(unique_values, unique_value_counts)
ax.set_ylabel("Frequency", size = 12)
ax.set_title("Type of author", size = 14)
plt.show()


# data is not balanced
# more than half the post do not explicitly state they are a professional, a parents/carer or if they have ASD
# more than half the posts come from the general population ( that most likely has not directly experienced ABA, be it as a client or a proveder)

In [ ]:
# Create Individual Columns for All Unique Values

def boolean_df(item_lists, unique_items):
# Create empty dict
    bool_dict = {}
    
    # Loop through all the tags
    for i, item in enumerate(unique_items):
        
        # Apply boolean mask
        bool_dict[item] = item_lists.apply(lambda x: item in x)
            
    # Return the results as a dataframe
    return pd.DataFrame(bool_dict)


In [ ]:
aba_posts_labeled_bool = boolean_df(
  item_lists = aba_posts_labeled["author"],
  unique_items = unique_values
)

In [ ]:
aba_posts_labeled_bool.head()

In [ ]:
# join 'Document' and 'sentiment' columns from aba_posts_labeled with aba_posts_labeled_bool
aba_posts_labeled_new = pd.concat([aba_posts_labeled[['Document', 'sentiment']], aba_posts_labeled_bool], axis=1)

aba_posts_labeled_new.info()
aba_posts_labeled_new.head()

In [ ]:
# save new dataset on google drive - needed to be imported in colab for running transformer models on GPU
aba_posts_labeled_new.to_csv('/Users/andreeanistor/Google Drive/My Drive/MS_DS_NLP_project/aba_posts_labeled_cleaned_full_set.csv', index=False)